In [1]:
import os
os.chdir("../")

In [ ]:
import numpy as np
import pandas as pd
import sys, os
from random import shuffle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from utils import *
import datetime
import matplotlib.pyplot as plt
import pickle
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
import gc
from torch_scatter import scatter_mean


In [ ]:
def return_omics_type(data):
    t = 0
    temp = []
    name = "_"
    for k, v in data.items():
        t = t + v
        if(v):
            temp.append(k)
    temp = tuple(temp)
    if t == 1:
        return "1omics", name.join(temp)
    if t == 2:
        return "2omics", name.join(temp)
    if t == 3:
        return "3omics", name.join(temp)
        
data_types = [
    {"ge":1, "mut":1, "meth":1},
    {"ge":1, "mut":1, "meth":0},
    {"ge":1, "mut":0, "meth":1},
    {"ge":0, "mut":1, "meth":1},
    {"ge":1, "mut":0, "meth":0},
    {"ge":0, "mut":1, "meth":0},
    {"ge":0, "mut":0, "meth":1},    
]
data_sets = ["all_test"]
# KHẮC PHỤC: Sử dụng torch.cuda.is_available() để tránh lỗi khi không có GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Define model

In [ ]:
from mol_bpe import Tokenizer
tokenizer = Tokenizer("vocab.txt")
FRAG_VOCAB_SIZE = tokenizer.num_subgraph_type()

# 1. ĐỊNH NGHĨA LỚP GINEncoder
class GINEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_layers=5):
        super(GINEncoder, self).__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_layers):
            # Cấu trúc GIN chuẩn: Linear -> ReLU -> Linear
            mlp = nn.Sequential(
                nn.Linear(in_dim if _ == 0 else hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
        return x

# 2. ĐỊNH NGHĨA CÁC MODULE KHÁC
# --- MODULE 1: Nhánh Tế bào (Multi-Omics 1D-CNN) ---
class MultiOmicsCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        def cnn_block(in_channels=1):
            return nn.Sequential(
                nn.Conv1d(in_channels, 32, kernel_size=7, padding=3), nn.ReLU(),
                nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
                nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
                nn.AdaptiveMaxPool1d(1), nn.Flatten()
            )
        self.ge_conv = cnn_block()
        self.mut_conv = cnn_block()
        self.meth_conv = cnn_block()
        self.fusion = nn.Linear(128 * 3, out_dim)

    def forward(self, ge, mut, meth):
        x = torch.cat([self.ge_conv(ge.unsqueeze(1)), 
                       self.mut_conv(mut.unsqueeze(1)), 
                       self.meth_conv(meth.unsqueeze(1))], dim=1)
        return self.fusion(x)

# --- MODULE 2, 3, 4: Mô hình tổng thể ---
class GraphOmicsSynergy_Fragment(nn.Module):
    def __init__(self, atom_in_dim=78, frag_in_dim=2300, hidden_dim=128):
        super().__init__()
        # Nhánh tế bào
        self.cell_encoder = MultiOmicsCNN(out_dim=hidden_dim)
        
        # Nhánh thuốc: Atom GIN (5 lớp), Frag GIN (2 lớp)
        self.atom_encoder = GINEncoder(atom_in_dim, hidden_dim, num_layers=5)
        self.frag_encoder = GINEncoder(frag_in_dim, hidden_dim, num_layers=2)
        
        # Alpha parameter (Intra-Drug Fusion - Sigmoid gate)
        self.alpha_param = nn.Parameter(torch.tensor(0.0))
        
        # Module 3: Attention Giao hoán
        self.attn_linear = nn.Linear(hidden_dim * 3, 1)
        
        # Module 4: Prediction Head
        self.predictor = nn.Sequential(nn.Linear(hidden_dim * 2, 512), nn.ReLU(), nn.Linear(512, 1))

    def forward(self, data):
        # 1. Intra-Drug Fusion (Module 2)
        def get_drug_rep(x_atom, edge_atom, x_frag, edge_frag, batch_atom, alpha):
            za = global_add_pool(self.atom_encoder(x_atom, edge_atom), batch_atom)
            zf = global_add_pool(self.frag_encoder(x_frag, edge_frag), batch_atom)
            return alpha * za + (1 - alpha) * zf

        alpha = torch.sigmoid(self.alpha_param)
        dA = get_drug_rep(data.x_1, data.edge_index_1, data.x_f1, data.edge_index_f1, data.x_1_batch, alpha)
        dB = get_drug_rep(data.x_2, data.edge_index_2, data.x_f2, data.edge_index_f2, data.x_2_batch, alpha)
        
        # 2. Cell Embedding (Module 1)
        cn = self.cell_encoder(data.target_ge, data.target_mut, data.target_meth)
        
        # 3. Inter-Drug Attention (Module 3 - Tính giao hoán)
        eA = torch.exp(F.leaky_relu(self.attn_linear(torch.cat([dA, cn, dB], dim=1))))
        eB = torch.exp(F.leaky_relu(self.attn_linear(torch.cat([dB, cn, dA], dim=1))))
        
        # Chuẩn hóa attention weights
        attn_A = eA / (eA + eB)
        attn_B = eB / (eA + eB)
        
        dpair = attn_A * dA + attn_B * dB
        
        # 4. Final Prediction (Module 4)
        output = self.predictor(torch.cat([dpair, cn], dim=1))
        
        # KHẮC PHỤC: Trả về prediction + attention weights cho 2 thuốc
        return output, attn_A, attn_B


In [ ]:
def train(model, device, train_loader, optimizer, epoch, log_interval, critation):
    print('Training on {} samples...'.format(len(train_loader.dataset)))
    model.train()
    avg_loss = []
    for batch_idx, data in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        # KHẮC PHỤC: Nhận 3 giá trị từ forward, chỉ dùng output cho loss
        output, _, _ = model(data)
        loss = critation(output, data.y.view(-1, 1).float().to(device))
        loss.backward()
        optimizer.step()
        avg_loss.append(loss.item())
        text = 'Train epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(epoch,
                                                                        batch_idx * len(data.x_1),
                                                                        len(train_loader.dataset),
                                                                        100. * batch_idx / len(train_loader),
                                                                        loss.item())
        print(text)
    return sum(avg_loss)/len(avg_loss)


In [ ]:
def predicting(model, device, loader, ats=False):
    model.eval()
    total_preds = torch.Tensor()
    total_labels = torch.Tensor()
    if ats:
        attn_drug_1 = torch.Tensor()
        attn_drug_2 = torch.Tensor()
        print('Make prediction for {} samples...'.format(len(loader.dataset)))
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
                # KHẮC PHỤC: Nhận 3 giá trị từ forward
                output, a_drug_1, a_drug_2 = model(data)
                total_preds = torch.cat((total_preds, output.cpu()), 0)
                total_labels = torch.cat((total_labels, data.y.view(-1, 1).cpu()), 0)
                attn_drug_1 = torch.cat((attn_drug_1, a_drug_1.cpu()), 0)
                attn_drug_2 = torch.cat((attn_drug_2, a_drug_2.cpu()), 0)
        return total_labels.numpy().flatten(), total_preds.numpy().flatten(), attn_drug_1.numpy().flatten(), attn_drug_2.numpy().flatten()
    else:
        print('Make prediction for {} samples...'.format(len(loader.dataset)))
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
                # KHẮC PHỤC: Lấy chỉ output khi không cần attention
                output, _, _ = model(data)
                total_preds = torch.cat((total_preds, output.cpu()), 0)
                total_labels = torch.cat((total_labels, data.y.view(-1, 1).cpu()), 0)
        return total_labels.numpy().flatten(), total_preds.numpy().flatten()


In [ ]:
def draw(list_, label, y_label, title):
    plt.figure()
    plt.plot(list_, label=label)
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(y_label)
    plt.legend()

    plt.savefig(title+".png")

In [ ]:
def return_ret(G, P):
    return [rmse(G,P),mse(G,P),pearson(G,P),spearman(G,P)]

def r2_rmse( g ):
            r2 =  mean_squared_error( g['synergy'], g['predict'] )
            count = len(g['synergy'])
            rmse = np.sqrt( mean_squared_error( g['synergy'], g['predict'] ) )
            return pd.Series( dict(  r2 = r2, rmse = rmse, count = count ) )
        
def get_top_data(r, df, top=10):
    # KHẮC PHỤC: Cập nhật để xử lý 4 return values từ predicting (G, P, attn_1, attn_2)
    G, P, attn_1, attn_2 = r
    attn_1 = np.array(attn_1)
    attn_2 = np.array(attn_2)
    top = top*2
    abs_error = np.abs(G-P)
    index_top = abs_error.argsort()[:]
    values = abs_error[index_top]
    df_top = df.iloc[index_top].copy()
    df_top["log_synergy"] = G[index_top]
    df_top["predict"] = P[index_top]
    df_top["abs_error"] = values
    df_top["attn_drug_1"] = attn_1[index_top]
    df_top["attn_drug_2"] = attn_2[index_top]
    return df_top


# Load data

In [ ]:
model_st = "DualView_GraphOmicsSynergy_Fragment"
dataset = 'GDSC'
train_batch = 1024
val_batch = 1024
test_batch = 1024
log_interval = 20

# Training

## Define paramters

In [12]:
lr = 0.001
num_epoch = 300
best_ret_test = None
print('Learning rate: ', lr)
print('Epochs: ', num_epoch)

Learning rate:  0.001
Epochs:  300


## Train model

In [ ]:
for data_type in data_types:
    for data_set in data_sets:
        
        data_path = f"data/split_data/{data_set}"
        data_processed_path = "data/split_data/{data_set}/processed/"
        model_st = "DualView_GraphOmicsSynergy_Fragment"
        dataset = 'GDSC'
        train_batch = 1024
        val_batch = 1024
        test_batch = 1024
        log_interval = 20

        print('\nrunning on ', model_st + '_' + dataset )
        train_data = TestbedDataset(root=data_path, dataset=dataset+"_"+'train_dc')
        val_data = TestbedDataset(root=data_path, dataset=dataset+"_"+'val_dc')
        test_data = TestbedDataset(root=data_path, dataset=dataset+"_"+'test_dc')
        
        loader_kwargs = {'follow_batch': ['x_1', 'x_2', 'x_f1', 'x_f2'], 'pin_memory': True}
        train_loader = DataLoader(train_data, batch_size=train_batch, shuffle=True, **loader_kwargs)
        val_loader = DataLoader(val_data, batch_size=val_batch, shuffle=False, **loader_kwargs)
        test_loader = DataLoader(test_data, batch_size=test_batch, shuffle=False, **loader_kwargs)

        print(device)
        model = GraphOmicsSynergy_Fragment(atom_in_dim=78, frag_in_dim=FRAG_VOCAB_SIZE, hidden_dim=128).to(device)
        try:
            model.atom_encoder.load_state_dict(torch.load('pretrained_weights/pretrained_atom_encoder.pt'))
            model.frag_encoder.load_state_dict(torch.load('pretrained_weights/pretrained_frag_encoder.pt'))
            print("Đã nạp trọng số Pre-trained thành công!")
        except FileNotFoundError:
            print("Cảnh báo: Không tìm thấy file trọng số, mô hình sẽ khởi tạo ngẫu nhiên.")

        lr = 0.001
        num_epoch = 300
        best_ret_test = None
        print('Learning rate: ', lr)
        print('Epochs: ', num_epoch)

        train_losses = []
        val_losses = []
        val_pearsons = []

        omics, name_omics = return_omics_type(data_type)
        save_path = "model/save_model/" + f"GIN_ADD_AT/{omics}/{name_omics}/{data_set}" + "/"
        print(save_path)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        best_mse = 1000
        best_pearson = 1
        best_epoch = -1

        best_val_losses = 10000

        ret_test_save = [1,1]

        model_file_name = save_path + 'best_model' + '.model'
        result_file_name = save_path + 'result_' + model_st + '_' + dataset +  '.csv'
        loss_fig_name = save_path + 'model_' + model_st + '_' + dataset + '_loss'
        pearson_fig_name = save_path + 'model_' + model_st + '_' + dataset + '_pearson'

        loss_fn = nn.MSELoss()
        for epoch in range(300):
            print('Epoch {} | Training on {} samples...'.format(epoch, len(train_loader.dataset)))
            avg_loss = []    
            train_loss = train(model, device, train_loader, optimizer, epoch+1, log_interval, loss_fn)
            #VAL:
            G, P = predicting(model, device, val_loader)
            ret = return_ret(G, P)

            train_losses.append(train_loss)
            val_losses.append(ret[1])
            val_pearsons.append(ret[2])

            if ret[1]<best_val_losses:
                best_val_losses = ret[1]
                # KHẮC PHỤC: Nhận 4 return values từ predicting với ats=True
                G_test, P_test, attn_1, attn_2 = predicting(model, device, test_loader, ats=True)
                ret_test_save = return_ret(G_test, P_test)
                print("RMSE all test improved")
                torch.save(model.state_dict(), model_file_name)

            with open(result_file_name,'w') as f:
                f.write('ret_test: '+','.join(map(str,ret_test_save)) +"\n")

            draw_loss(train_losses, val_losses, loss_fig_name)
            draw_pearson(val_pearsons, pearson_fig_name)

            torch.save(model.state_dict(), model_file_name)
            log = [
                train_losses, val_losses
                ]

            with open(save_path+ "/log.pickle", "wb") as f:
                pickle.dump(log, f)

        data_split_path = f"data/split_data/{data_set}/"
        train_dc = pd.read_csv(data_split_path+"train_dc.csv")
        test_dc = pd.read_csv(data_split_path+"test_dc.csv")
        val_dc = pd.read_csv(data_split_path+"val_dc.csv")

        model.load_state_dict(torch.load(model_file_name))
        
        # KHẮC PHỤC: Nhận 4 return values từ predicting
        G_test, P_test, attn_1, attn_2 = predicting(model, device, test_loader, ats=True)
        result_dict = {
            "test": ((G_test, P_test, attn_1, attn_2), test_dc),
        }

        for key, value in tqdm(result_dict.items()):
            temp = get_top_data(value[0], value[1])
            temp.to_csv(save_path+"detail_result.csv", index=False)
        
        # BÂY GIỜ MỚI DEL MODEL
        del model
        torch.cuda.empty_cache() 
        gc.collect()
